# 06 日志与主从复制

## 八股问答

### 1. MySQL 三种日志的作用？

| 日志 | 层次 | 作用 |
| --- | --- | --- |
| redo log | InnoDB 引擎层，物理日志 | 崩溃恢复，重放已提交的修改 |
| undo log | InnoDB 引擎层，逻辑日志 | 回滚、MVCC 版本链 |
| binlog | MySQL 服务层，逻辑日志 | 主从复制、基于时间点的恢复 |

面试关键点：redo 和 undo 是引擎的，binlog 是 Server 的；redo 记录“页改成了
什么”，binlog 记录“一条语句/一行变成了什么”。

### 2. undo log 的字段结构？如何回滚？

undo log 记录的是反向操作：

- INSERT 记删除标记，回滚时删除该行。
- DELETE 记被删行的完整旧值，回滚时恢复。
- UPDATE 记修改前的旧版本，回滚时把旧版本覆盖回去。

每行有隐藏列 `DB_ROLL_PTR` 指向自己的上一个 undo 版本，形成版本链。回滚时
事务沿自己产生的 undo 记录反向执行，不需要扫描全表；同一版本链也被 MVCC
快照读复用。

### 3. redo log 什么时候刷盘？

由参数 `innodb_flush_log_at_trx_commit` 决定：

| 值 | 行为 | 可靠性 |
| --- | --- | --- |
| 1（默认） | 每次事务提交都刷盘 | 最可靠，最多丢一个事务 |
| 0 | 每秒刷盘 | 崩溃可能丢最近 1 秒事务 |
| 2 | 提交时写 OS cache，每秒刷盘 | 操作系统崩溃丢 1 秒，MySQL 崩溃不丢 |

另外 redo log 是循环写入的固定文件组（8.0.30 后由
`innodb_redo_log_capacity` 管理，默认 100MB）。

### 4. redo log 写满了怎么办？

redo log 循环写，写满时不能覆盖还没落盘（未 checkpoint）的日志。此时 InnoDB
会强制把对应脏页刷到磁盘，推进 checkpoint，然后才能继续写日志。表现为写入
暂时变慢，所以“redo 写满”通常意味着刷脏速度跟不上写入速度。

### 5. redo log 和 binlog 的区别？

| 维度 | redo log | binlog |
| --- | --- | --- |
| 层次 | InnoDB 引擎 | MySQL 服务层 |
| 内容 | 物理页修改 | 逻辑 SQL / 行变更 |
| 写入时机 | 事务执行中持续写 | 事务提交时写 |
| 存储 | 固定大小循环写 | 追加写，可归档 |
| 用途 | 崩溃恢复 | 主从复制、恢复 |

### 6. binlog 什么时候刷盘？

由 `sync_binlog` 控制：

- 0：由操作系统决定，性能最好，崩溃可能丢日志。
- 1：每次事务提交都 fsync，最可靠，性能开销最大。
- N：每 N 次提交刷一次，折中。

### 7. 主从复制的实现方法？

经典异步复制三步：

1. 主库：事务提交前把变更写入 binlog。
2. 从库：IO 线程拉取主库 binlog，写入自己的中继日志 relay log。
3. 从库：SQL 线程顺序回放 relay log，应用变更。

binlog 三种格式：

- STATEMENT：记录 SQL，日志小，但函数和不确定语句在主从不一致。
- ROW（默认）：记录行前后镜像，最准确，日志较大。
- MIXED：默认 STATEMENT，遇到不确定语句自动切 ROW。

延迟优化：半同步复制（至少一个从库确认）、并行复制（多线程回放）、
GTID（自动定位断点）、把从库查询分流。

### 8. UPDATE 语句的执行过程？

1. 连接器校验权限，解析器解析 SQL。
2. 优化器生成执行计划，执行器按计划定位目标行（当前读）。
3. 对目标行加 X 锁。
4. 写 undo log 保存旧版本。
5. 在 Buffer Pool 中修改数据页，页标记为脏页。
6. 写 redo log（prepare 阶段）。
7. 写 binlog。
8. redo log 标记 commit（两阶段提交完成）。
9. 提交事务、释放锁；脏页由后台线程择机刷盘。

### 9. 为什么两阶段提交？过程是什么？

如果 redo 和 binlog 不原子，崩溃后可能出现：主库恢复后有这条数据、从库没有
（redo 有 binlog 无），或者相反。两阶段提交保证两者一致：

1. prepare：redo log 写入并标记 prepare。
2. 写 binlog 并刷盘。
3. commit：redo log 标记 commit。

崩溃恢复时按状态判断：

- redo prepare + binlog 已写：事务有效，补 commit。
- redo prepare + binlog 未写：事务回滚。
- redo commit：直接生效。

### 10. 提交时异常重启会发生什么？

取决于异常点：

- binlog 写之前崩溃：redo 只有 prepare，恢复时回滚，主从都没有。
- binlog 写完后崩溃：恢复时看到 prepare + binlog 完整，补 commit，
  主从一致。

这正是两阶段提交要解决的“主从不一致”问题。

### 11. MySQL 的 I/O 很高，如何优化？

先定位是读 I/O 还是写 I/O：

- 读 I/O：加索引减少扫描、扩大 `innodb_buffer_pool_size` 提高命中率、
  用覆盖索引、优化深分页。
- 写 I/O：批量写入减少提交次数、调 `innodb_flush_log_at_trx_commit`
  与 `sync_binlog`（要接受相应可靠性损失）、避免频繁更新大字段、
  避免每次 UPDATE 都全表扫描（产生大量随机写）、控制长事务。
- 容量：`SHOW ENGINE INNODB STATUS` 看脏页和刷盘速率，`iostat` 看磁盘
  利用率，区分是磁盘慢还是刷盘量太大。


## 面试话术

“写入链路是：加锁 → 写 undo → 改 Buffer Pool 脏页 → 写 redo（prepare）→
写 binlog → redo commit。两阶段提交解决崩溃时 redo 和 binlog 不一致的问题，
恢复规则是 binlog 完整就补 commit，否则回滚。主从靠 binlog + relay log 回放，
延迟优化用半同步、并行复制和 GTID。I/O 高先分清读还是写：读优化索引和
Buffer Pool，写优化批量提交和刷盘参数，但调整 innodb_flush_log_at_trx_commit
前必须和业务确认能接受的数据丢失窗口。”


## 自查清单

- [ ] 能说清 redo/undo/binlog 的层次、内容和用途
- [ ] 能画出 UPDATE 从加锁到两阶段提交的完整时序
- [ ] 能解释“prepare + binlog 完整”和“prepare + binlog 缺失”的恢复结果
- [ ] 能说出主从复制三个线程和 binlog 三种格式
- [ ] 能在终端看到 binlog 事件和 redo/binlog 刷盘参数
